# CP2 · Notebook 05 — Depth Anything v2

## Objetivo

Aplicar **Depth Anything v2 small** a las mismas ~21 imágenes, comparar visual y cuantitativamente con MiDaS, e identificar dónde aporta y dónde no.

**Tiempo estimado**: 8 min.

## 0. Setup + carga del modelo

In [ ]:
import time, json, numpy as np, torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from tqdm import tqdm

CP1_DATA   = Path('../../CP1-carriles/datasets/lanes-subset')
CP2_EXTRAS = Path('../datasets/cp2-depth-extras')
OUT_DIR    = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True)

DA_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
da_processor = AutoImageProcessor.from_pretrained(DA_ID)
da_model = AutoModelForDepthEstimation.from_pretrained(DA_ID).eval()
print(f'✅ DA v2 small ({sum(p.numel() for p in da_model.parameters())/1e6:.1f} M params)')

## 1. Inferencia sobre todas las imágenes

In [ ]:
def da_infer(image_pil):
    t0 = time.perf_counter()
    inputs = da_processor(images=image_pil, return_tensors='pt')
    with torch.no_grad():
        out = da_model(**inputs).predicted_depth
    return out[0].numpy(), (time.perf_counter() - t0) * 1000

# Warmup
sample = next((CP1_DATA / 'easy').glob('*.png'))
_ = da_infer(Image.open(sample).convert('RGB'))

image_set = []
for cat in ['easy', 'medium', 'hard']:
    for p in sorted((CP1_DATA / cat).glob('*.png')):
        image_set.append({'path': p, 'origin': 'CP1', 'category': cat, 'name': p.name})
for p in sorted(CP2_EXTRAS.glob('*.jpg')) + sorted(CP2_EXTRAS.glob('*.png')):
    image_set.append({'path': p, 'origin': 'CP2_extras', 'category': 'challenge', 'name': p.name})

results = []
for item in tqdm(image_set, desc='DA v2'):
    img = Image.open(item['path']).convert('RGB')
    depth, ms = da_infer(img)
    results.append({**item, 'depth': depth, 'time_ms': ms})

times = [r['time_ms'] for r in results]
print(f'\nDA v2: tiempo medio {np.mean(times):.0f} ms · p95 {np.percentile(times, 95):.0f} ms')

## 2. Comparativa MiDaS vs DA v2 — grid

In [ ]:
# Cargar MiDaS results del notebook 03
with open(OUT_DIR / '03_midas_summary.json') as f:
    midas_summary = json.load(f)
midas_index = {e['name']: e for e in midas_summary['per_image']}
midas_depths = {e['name']: np.load(OUT_DIR / '03_midas_depth' / e['npy_file']) for e in midas_summary['per_image']}

# Pick 4 imágenes representativas, una de cada categoría
samples = []
for cat in ['easy','medium','hard','challenge']:
    matches = [r for r in results if r['category'] == cat]
    if matches: samples.append(matches[0])

fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5*len(samples)))
for row, r in enumerate(samples):
    img = np.array(Image.open(r['path']).convert('RGB'))
    axes[row, 0].imshow(img); axes[row, 0].set_title(f"{r['category']} · {r['name']}")
    axes[row, 0].axis('off')
    midas_depth = midas_depths.get(r['name'])
    if midas_depth is not None:
        im1 = axes[row, 1].imshow(midas_depth, cmap='plasma')
        axes[row, 1].set_title(f"MiDaS  ({midas_index[r['name']]['time_ms']:.0f} ms)")
        plt.colorbar(im1, ax=axes[row, 1], fraction=0.046)
    axes[row, 1].axis('off')
    im2 = axes[row, 2].imshow(r['depth'], cmap='plasma')
    axes[row, 2].set_title(f"DA v2  ({r['time_ms']:.0f} ms)")
    plt.colorbar(im2, ax=axes[row, 2], fraction=0.046)
    axes[row, 2].axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / '05_compare_grid.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Diferencia absoluta normalizada

Para una comparación cuantitativa, **normalizamos** cada depth a [0, 1] (porque MiDaS y DA tienen rangos absolutos distintos) y miramos donde discrepan.

In [ ]:
def normalize(d):
    return (d - d.min()) / (d.max() - d.min() + 1e-6)

fig, axes = plt.subplots(len(samples), 2, figsize=(14, 5*len(samples)))
if len(samples) == 1: axes = axes.reshape(1, 2)
for row, r in enumerate(samples):
    midas_norm = normalize(midas_depths[r['name']])
    # Asegurar mismo shape — los modelos pueden output a tamaños distintos
    if midas_norm.shape != r['depth'].shape:
        import cv2
        midas_norm = cv2.resize(midas_norm, (r['depth'].shape[1], r['depth'].shape[0]))
    da_norm = normalize(r['depth'])
    diff = np.abs(midas_norm - da_norm)
    img = np.array(Image.open(r['path']).convert('RGB'))
    axes[row, 0].imshow(img); axes[row, 0].set_title(f"{r['category']} · {r['name']}"); axes[row, 0].axis('off')
    im = axes[row, 1].imshow(diff, cmap='hot', vmin=0, vmax=0.5)
    axes[row, 1].set_title(f'|MiDaS − DA v2| normalizado  ·  max={diff.max():.2f}  mean={diff.mean():.3f}')
    axes[row, 1].axis('off'); plt.colorbar(im, ax=axes[row, 1], fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / '05_diff_normalized.png', dpi=100, bbox_inches='tight')
plt.show()
print('Las zonas más calientes son donde los modelos discrepan más. Apunta cuál — para 07_preguntas.md.')

## 4. Comparativa de tiempos

In [ ]:
midas_times = [e['time_ms'] for e in midas_summary['per_image']]
da_times    = times

fig, ax = plt.subplots(figsize=(10, 4))
bp = ax.boxplot([midas_times, da_times], tick_labels=['MiDaS small', 'DA v2 small'], patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['#4a6fa5', '#DA4544']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('Tiempo de inferencia (ms)')
ax.set_title('Tiempo CPU MiDaS small vs Depth Anything v2 small')
plt.tight_layout(); plt.savefig(OUT_DIR / '05_times.png', dpi=100, bbox_inches='tight'); plt.show()

print(f'MiDaS:  media {np.mean(midas_times):.0f} ms  p95 {np.percentile(midas_times, 95):.0f} ms')
print(f'DA v2:  media {np.mean(da_times):.0f} ms  p95 {np.percentile(da_times, 95):.0f} ms')
ratio = np.mean(da_times) / np.mean(midas_times)
print(f'\nDA v2 / MiDaS  = {ratio:.2f}×')

## 5. Guardar resumen DA v2

In [ ]:
DEPTH_DIR = OUT_DIR / '05_da_depth'
DEPTH_DIR.mkdir(exist_ok=True)

summary = {
    'model': DA_ID,
    'mean_time_ms': float(np.mean(da_times)),
    'median_time_ms': float(np.median(da_times)),
    'p95_time_ms': float(np.percentile(da_times, 95)),
    'total_images': len(results),
    'speedup_vs_midas': float(np.mean(midas_times) / np.mean(da_times)),
    'per_image': [],
}
for r in results:
    npy_name = f"{r['origin']}_{r['category']}_{r['name']}.npy"
    np.save(DEPTH_DIR / npy_name, r['depth'])
    summary['per_image'].append({
        'name': r['name'], 'category': r['category'], 'origin': r['origin'],
        'time_ms': float(r['time_ms']),
        'depth_min': float(r['depth'].min()),
        'depth_max': float(r['depth'].max()),
        'depth_mean': float(r['depth'].mean()),
        'npy_file': npy_name,
    })

with open(OUT_DIR / '05_da_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ DA v2 outputs guardados en {DEPTH_DIR}/')
print(f'✅ Resumen en {OUT_DIR / "05_da_summary.json"}')

## 6. Reflexión

Para `07_preguntas.md`, apunta:

1. ¿Qué modelo da depth maps más "limpios" en escenarios típicos (easy/medium)?
2. ¿Cuál se ve más afectado por las imágenes `challenge` (reflejos, cielo, túnel)?
3. ¿Compensaría el coste extra de uno sobre el otro?

Ve a `06_failure_modes.ipynb` para profundizar en dónde fallan ambos.